<a href="https://colab.research.google.com/github/Shacxify/prompt-engineering-exercises/blob/main/03_self_reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 3 - Self-Reflection Prompt for Improving Output

**Cash Johnson | BUS4 118S - Agentic AI for Business | Prof. Haubrich**

**Tools used:** Google Colab + Google Gemini API (`google-genai` SDK). The setup cell picks
the model at runtime from what the key can actually call, and prints which one it landed on,
because free-tier quota is metered per model per day and model names get retired.

**Goal:** have the model critique its own summary against explicit written criteria and produce a
revised version, then verify the improvement instead of taking the model's word for it.

**Source:** a Q3 operations memo for VNTG OS, the vintage consignment app I built for BUS4 110B.
**Audience for the summary:** the store owner. Non-technical, cares about money, staffing, and what
to do Monday. Does not care about platform metrics and does not know what "sell-through" means.

---

### Structure

| Section | What happens |
|---|---|
| 2 | **BEFORE** - a deliberately loose summary prompt, the kind most people actually write |
| 3 | The self-reflection prompt: five named criteria, each with a pass condition |
| 4 | **AFTER** - the revised summary the critique produced |
| 5 | Python checks: word count, bullet count, jargon, and every number traced back to the memo |
| 6 | A separate grader call scores both versions on the module's three evaluation metrics plus the three task constraints |
| 7 | RSIP: a second reflection pass, to test whether recursion keeps paying |
| 8 | Before and after side by side, and what actually changed |

The measurement in sections 5 and 6 is the part that matters. "The second one looks better" is not
evidence, and a model asked whether it improved its own work will always say yes.

### Techniques from the module used in this notebook

| Module technique | Where it shows up here |
|---|---|
| **Self-reflection prompting** | section 3: the model evaluates its own output against a fixed checklist before rewriting it |
| **RSIP (recursive self-improvement)** | section 7: a second reflection pass on the already-revised summary, to find out whether recursion keeps paying |
| **Back-and-forth refinement** | the before/critique/after sequence is iterative prompting inside one notebook |
| **System prompt vs user prompt** | the editor persona is the SYSTEM prompt, the memo, draft, and criteria are the USER prompt |
| **Role-based prompting** | "a demanding editor", and a separate "grader" persona that scores both versions |
| **Clarity and specificity** | five criteria written as checkable conditions, not adjectives |
| **Content structuring** | a required two-header output format, which is what makes the revision extractable in code |
| **Negative prompting** | an explicit ban list of jargon, and "do not compute, combine, round, or annualize anything" |
| **AI evaluation metrics** | section 6 scores both versions on the module's three metrics, relevance, coherence, and accuracy, alongside the three task constraints |
| **Hallucination check** | section 5 traces every number in the summary back to the source memo, because invented figures are the failure mode summaries have |
| **Temperature** | 0.3 on the first draft to let it behave like a normal unconstrained summary, 0 everywhere a result has to be reproducible |

In [10]:
!pip install -q -U google-genai

from google import genai
from google.genai import types
from google.genai import errors as genai_errors
import json, time, re

try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Gemini API key: ')

client = genai.Client(api_key=API_KEY)

# ---------------------------------------------------------------------------
# Free-tier quota is metered PER PROJECT, PER MODEL, PER DAY. So each of the
# three notebooks pins a DIFFERENT model and gets its own daily allowance
# instead of all three draining one bucket. This notebook takes slot 2.
#
# Model names change and not every listed model is callable on every key, so
# rather than hard-coding one, list what the key can see and probe until one
# actually answers. Set PIN below to override.
# ---------------------------------------------------------------------------
PIN = None          # e.g. 'gemini-3.6-flash' to force a specific model
SLOT = 2       # which model in the preference order this notebook claims

def _candidates():
    names = []
    for m in client.models.list():
        actions = getattr(m, 'supported_actions', None) or []
        if actions and 'generateContent' not in actions:
            continue
        n = m.name.replace('models/', '')
        if 'embedding' in n or 'imagen' in n or 'veo' in n or 'tts' in n:
            continue
        if 'flash' in n or 'pro' in n:
            names.append(n)
    names.sort(key=lambda n: (0 if 'flash-lite' in n else 1 if 'flash' in n else 2, n))
    return names

def _probe(name):
    try:
        r = client.models.generate_content(
            model=name, contents='say ok',
            config=types.GenerateContentConfig(temperature=0))
        return bool((r.text or '').strip())
    except genai_errors.APIError as e:
        print(f'  {name}: unavailable ({getattr(e, "code", "?")})')
        return False

MODEL = None
if PIN:
    MODEL, pool = PIN, [PIN]
else:
    pool = _candidates()
    print(f'{len(pool)} candidate models on this key')
    for name in pool[SLOT:] + pool[:SLOT]:
        if _probe(name):
            MODEL = name
            break
assert MODEL, f'no callable model found. Candidates were: {pool[:8]}'
print('Using:', MODEL)

CALLS = 0
RETRY_ON = {500, 502, 503, 504}

# Free tier meters TWO different quotas and they need opposite responses:
#   per minute (RPM 5-10) -> transient, wait it out and carry on
#   per day    (RPD 20)   -> gone until reset, no amount of waiting helps
# The 429 body names which one via quotaId, so read it instead of guessing.
MIN_INTERVAL = 13.0     # seconds between calls, keeps us under ~5 RPM
_last_call = [0.0]

def _pace():
    gap = time.time() - _last_call[0]
    if gap < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - gap)
    _last_call[0] = time.time()

def _quota_kind(err):
    blob = str(getattr(err, 'message', '') or err)
    if 'PerDay' in blob:
        return 'day'
    if 'PerMinute' in blob:
        return 'minute'
    return 'unknown'

def _retry_delay(err, default=30.0):
    m = re.search(r"retryDelay'?\s*:\s*'?(\d+(?:\.\d+)?)s", str(err))
    return float(m.group(1)) + 2 if m else default

def _call(model, contents, cfg, retries=4):
    """Every API call goes through here so the run is paced, counted, and survives a blip."""
    global CALLS
    for attempt in range(retries):
        _pace()
        try:
            r = client.models.generate_content(model=model, contents=contents, config=cfg)
            CALLS += 1
            return r
        except genai_errors.APIError as e:
            code = getattr(e, 'code', None)
            if code == 429:
                kind = _quota_kind(e)
                if kind == 'day':
                    raise RuntimeError(
                        f'Daily quota gone for {model} after {CALLS} calls this session. '
                        f'The cap is per model per day, so set PIN to another model and re-run '
                        f'from this cell, or wait for reset. See https://ai.dev/rate-limit') from e
                if attempt == retries - 1:
                    raise
                wait = _retry_delay(e)
                print(f'  [429 {kind}] over the per-minute limit, waiting {wait:.0f}s')
                time.sleep(wait)
                continue
            if code not in RETRY_ON or attempt == retries - 1:
                raise
            wait = 3 * 2 ** attempt
            print(f'  [{code}] model busy, retrying in {wait}s')
            time.sleep(wait)

def ask(user_prompt, system=None, temperature=0.0, json_mode=False, model=None):
    kwargs = {'temperature': temperature}
    if system is not None:
        kwargs['system_instruction'] = system
    if json_mode:
        kwargs['response_mime_type'] = 'application/json'
    resp = _call(model or MODEL, user_prompt, types.GenerateContentConfig(**kwargs))
    return (resp.text or '').strip()


print('helper ready | model:', MODEL, '| calls so far:', CALLS)

27 candidate models on this key
  gemini-3.1-flash-lite-image: unavailable (429)
Using: gemini-3.1-flash-lite-preview
helper ready | model: gemini-3.1-flash-lite-preview | calls so far: 0


In [11]:
# =====================================================================
# THE SOURCE DOCUMENT, and the BEFORE summary from a loose prompt
# =====================================================================
# The loose prompt is the one most people write. It names the task and
# nothing else: no audience, no length, no format, no rule about where
# numbers may come from. Everything it leaves unsaid, the model fills in.

MEMO = """TO:      Store Ownership
FROM:    VNTG OS Platform Operations
RE:      Q3 FY26 consignor performance, intake throughput, and payout operations
DATE:    September 30, 2026

INTAKE VOLUME AND SLA
Q3 closed with 1,842 items accepted into intake across the 1st Street and Santana Row drop
points, up from 1,494 in Q2, a 23.3% increase quarter over quarter. Of those, 1,611 were
photographed, priced, and listed inside the ten business day intake SLA, putting on-time
listing at 87.5% against a target of 92%. The 231 late items averaged 14.2 business days from
drop-off to live listing. Late listings cluster almost entirely in the two weeks following each
of the three campus move-in weekends, when drop-off volume ran 41% above trailing average while
intake staffing stayed flat at two photographers per store.

SALES AND PAYOUTS
Gross merchandise value for the quarter was $96,412 across 1,203 sold items, an average sale
price of $80.14. Consignor payouts totaled $58,247, a blended consignor share of 60.4%,
consistent with the standard 60/40 split plus the 65% tier that applies to items priced at $200
and above. 87 items cleared the $200 tier this quarter, up from 54 in Q2.

SELL-THROUGH
Of the 1,611 items listed, 1,203 sold inside the quarter, a 74.7% sell-through rate. Items
priced under $45 sold at 81.2%, items between $45 and $120 at 76.9%, and items above $120 at
52.4%. Median days to sale was 19, up from 16 in Q2. The pricing team ran a limited test in
August, marking down 140 items that had sat past 45 days by 20%. 96 of those 140 cleared inside
three weeks, which is a 68.6% clearance rate on aged stock that had previously been moving at
roughly 22%.

RETURNS
68 items were returned inside the 7-day buyer return window, 5.7% of sold items, up from 4.1%
in Q2. 41 of the 68 return reasons cited fit or measurement discrepancy. Measurements are
entered by hand at intake and are not validated against anything. Each return also reverses a
payout line that has usually already been queued, which is the direct cause of the payout delays
described below.

CONSIGNOR RETENTION
312 active consignors at quarter end, 74 new this quarter, 38 churned, defined as no drop-off
within 90 days of their last payout. Churn concentrates hard among small first payouts: 26 of
the 38 churned consignors had a first payout under $25. Median first payout for retained
consignors was $61.

PAYOUT OPERATIONS
Payouts run twice monthly, on the 1st and the 15th. Three of the six Q3 cycles ran late, by an
average of 2.4 days. All three delays trace to manual reconciliation of returned items against
payout lines that were already queued. Ops estimates 6 to 8 staff hours per late cycle spent on
that reconciliation by hand.

SUPPORT LOAD
417 support tickets in Q3, 61% consignor-side. Top categories were intake status at 38%, payout
amount questions at 27%, and item condition disputes at 11%. Average first response time was
9.4 hours against a stated 8 hour target.

RECOMMENDATIONS
1. Add a third photographer at 1st Street for the four weeks following each move-in weekend.
   Estimated cost $3,100 per cycle; would have covered roughly 180 of the 231 late items.
2. Add measurement validation at intake, either a required second entry or a template per
   garment category. Targets the 41 fit-related returns.
3. Automate return reconciliation against the payout queue before the cycle runs. Removes the
   single named cause of all three late payout cycles and 6 to 8 hours of manual work each.
4. Test a first-payout floor or payout bundling for consignors under $25, aimed at the 26
   churned consignors in that band.
"""

print(f"source memo: {len(MEMO.split())} words")

PROMPT_BEFORE = """Summarize the following memo for the business owner.

{memo}"""

summary_before = ask(PROMPT_BEFORE.format(memo=MEMO), temperature=0.3)
print(summary_before)
print('\n---')
print('word count:', len(summary_before.split()))

source memo: 608 words
### **Executive Summary: Q3 FY26 Performance Review**

Q3 saw strong growth in volume and sales, but operational bottlenecks—specifically during peak periods and return processing—are impacting efficiency and consignor retention.

#### **Key Performance Metrics**
*   **Growth:** Intake volume increased by 23.3% (1,842 items). Gross Merchandise Value (GMV) reached $96,412.
*   **Efficiency:** 87.5% of items were listed within the 10-day SLA (target: 92%). Delays are concentrated during post-move-in surges.
*   **Sales:** Strong sell-through rate of 74.7%. A markdown test on aged inventory successfully cleared 68.6% of stagnant stock.
*   **Returns:** Return rate rose to 5.7%, primarily due to fit/measurement discrepancies.
*   **Operations:** Payouts were delayed in 50% of cycles due to manual reconciliation of returns.

#### **Critical Challenges**
1.  **Staffing Gaps:** Current intake staffing cannot handle the 41% volume spike following campus move-in weekends,

## 3. The self-reflection prompt (SYSTEM + USER)

Five criteria, each written as a checkable condition rather than an adjective. "Make it better"
gives the model nothing to test against; "every number must appear verbatim in the memo" does.

| # | Criterion | Pass condition |
|---|---|---|
| 1 | Factual accuracy | every claim traceable to a specific line of the memo |
| 2 | Numeric integrity | every figure appears verbatim in the memo, no arithmetic of its own |
| 3 | Length | 120 words or fewer, counted |
| 4 | Format | exactly 3 bullets, each leading with the action, plus one closing line naming the costliest single problem |
| 5 | Audience fit | no operations jargon; the owner's vocabulary, not the platform's |

The prompt also forces the critique to **quote the offending text** for any criterion it fails. That
single constraint is what stopped it from writing "could be more concise" and calling that a review.

In [12]:
REFLECT_SYSTEM = """You're a hard editor checking a draft against a checklist. You don't
praise and you don't soften. If something passes, say PASS and move on in one line."""

PROMPT_REFLECT = """Below is a memo, a summary someone wrote of it, and what the summary
is supposed to do. Tear into the summary, then rewrite it.

WHAT IT HAS TO DO
1. ACCURACY - every claim traces back to a line in the memo. No inference, no
   conclusions the memo doesn't state.
2. NUMBERS - every figure in the summary appears in the memo word for word. Don't
   compute, combine, round, or annualize anything. If it's not in the memo, it doesn't
   go in.
3. LENGTH - 120 words maximum. Count it before you hand it over.
4. FORMAT - three bullets. Each one leads with what the owner should do, then why, then
   the single number behind it. After the bullets, one line naming the costliest problem
   and its number. Nothing else, no headers, no preamble.
5. AUDIENCE - the reader owns the store and isn't technical. Don't use SLA, GMV, gross
   merchandise value, sell-through, throughput, churn, blended, reconciliation, cycle,
   variance, YoY, or QoQ. Say it the way he'd say it.

MEMO
{memo}

DRAFT SUMMARY
{draft}

Answer in two parts, with these headers exactly:

CRITIQUE:
One line per item, numbered 1 to 5: PASS or FAIL, what's wrong, and quote the exact text
that's wrong. Give me the real word count on 3. On 2, list every number you couldn't find
in the memo.

REVISED SUMMARY:
The rewrite, hitting all five. Nothing after it."""

reflection = ask(
    PROMPT_REFLECT.format(memo=MEMO, draft=summary_before),
    system=REFLECT_SYSTEM,
)
print(reflection)

parts = re.split(r'REVISED SUMMARY:\s*', reflection, maxsplit=1)
assert len(parts) == 2, 'header not found - check the model followed the output format'
critique, summary_after = parts[0].replace('CRITIQUE:', '').strip(), parts[1].strip()

print(summary_after)
print('\n---')
print('word count:', len(summary_after.split()))

CRITIQUE:
1. FAIL: Includes conclusions not in memo (e.g., "strong growth," "operational bottlenecks").
2. FAIL: Includes numbers not in memo: 50%, 4-week, 4-week.
3. FAIL: Word count is 268 words.
4. FAIL: Does not follow the three-bullet format or the required structure.
5. FAIL: Uses forbidden terms: GMV, SLA, churn.

REVISED SUMMARY:
* Hire a third photographer at 1st Street during move-in surges to prevent listing delays, which affected 231 items.
* Require a second measurement entry or template at intake to fix fit issues, which caused 41 returns.
* Automate return checks against payouts to stop late payments, which occurred in 3 cycles.

The costliest problem is late listings at 231.
* Hire a third photographer at 1st Street during move-in surges to prevent listing delays, which affected 231 items.
* Require a second measurement entry or template at intake to fix fit issues, which caused 41 returns.
* Automate return checks against payouts to stop late payments, which occurred i

## 5. Measured, not asserted

Four checks. The numeric one is the important one: it pulls every figure out of the summary and
looks for it verbatim in the memo. A figure that is not there is either invented or calculated, and
criterion 2 bans both.

In [13]:
JARGON = ['sla', 'gmv', 'gross merchandise', 'sell-through', 'sell through', 'throughput',
          'churn', 'blended', 'reconciliation', 'reconcile', 'qoq', 'yoy', 'variance']

NUM = re.compile(r'\d[\d,]*(?:\.\d+)?')

def numbers_in(text):
    return {m.group().replace(',', '').rstrip('.') for m in NUM.finditer(text)}

MEMO_NUMBERS = numbers_in(MEMO)

def check(summary, label):
    words = len(summary.split())
    bullets = len([l for l in summary.splitlines()
                   if l.strip().startswith(('-', '*', '\u2022')) or re.match(r'^\s*\d[\.\)]', l)])
    jargon_hits = sorted({j for j in JARGON if j in summary.lower()})
    unsourced = sorted(numbers_in(summary) - MEMO_NUMBERS)
    return {
        'version': label,
        'words (<=120)': words,
        'bullets (==3)': bullets,
        'jargon hits (==0)': len(jargon_hits),
        'jargon words': ', '.join(jargon_hits) or 'none',
        'numbers not in memo (==0)': len(unsourced),
        'which numbers': ', '.join(unsourced) or 'none',
    }

rows = [check(summary_before, 'BEFORE'), check(summary_after, 'AFTER')]

keys = list(rows[0].keys())[1:]
w = max(len(k) for k in keys) + 2
print(f"{'CHECK':<{w}}{'BEFORE':>12}{'AFTER':>12}")
print('-' * (w + 24))
for k in keys:
    print(f'{k:<{w}}{str(rows[0][k]):>12}{str(rows[1][k]):>12}')

# Any number in a summary that is not verbatim in the memo is a flag for a human to check,
# not automatic proof of error - a correctly copied number written a different way lands here too.
for r in rows:
    print(f"{r['version']}: {r['which numbers']}")

GRADER_SYSTEM = 'You grade summaries. JSON only, no explaining outside the JSON.'

GRADER = """Score both summaries 1 to 5 on each of these. 5 is full marks, 1 is a total
miss. Judge them against the memo and nothing else. The order they show up in means nothing.

CRITERIA - the three standard AI evaluation metrics:
relevance   - how closely the summary serves what was asked: what the store owner must decide
coherence   - logical flow and internal consistency
accuracy    - factually correct against the memo, with no figure the memo does not state

CRITERIA - the three task constraints:
length      - 120 words or fewer
format      - exactly 3 action-first bullets plus one closing line naming the costliest problem
audience    - plain language for a non-technical store owner, no operations jargon

MEMO
{memo}

SUMMARY_X
{a}

SUMMARY_Y
{b}

Return ONLY:
{{"X": {{"relevance": n, "coherence": n, "accuracy": n, "length": n, "format": n, "audience": n,
        "note": "under 20 words"}},
 "Y": {{"relevance": n, "coherence": n, "accuracy": n, "length": n, "format": n, "audience": n,
        "note": "under 20 words"}}}}"""

def grade(a, b):
    return json.loads(ask(GRADER.format(memo=MEMO, a=a, b=b), system=GRADER_SYSTEM, json_mode=True))

CRIT = ['relevance', 'coherence', 'accuracy', 'length', 'format', 'audience']

def score_table(scores, col_a='BEFORE', col_b='AFTER'):
    print(f"{'CRITERION':<12}{col_a:>9}{col_b:>9}{'DELTA':>8}")
    print('-' * 38)
    for c in CRIT:
        x, y = scores['X'][c], scores['Y'][c]
        mark = '  <- module metric' if c in ('relevance', 'coherence', 'accuracy') else ''
        print(f'{c:<12}{x:>9}{y:>9}{y - x:>+8}{mark}')
    print('-' * 38)
    tx, ty = sum(scores['X'][c] for c in CRIT), sum(scores['Y'][c] for c in CRIT)
    print(f"{'TOTAL /30':<12}{tx:>9}{ty:>9}{ty - tx:>+8}")
    return tx, ty

scores = grade(summary_before, summary_after)
total_before, total_after = score_table(scores)
print('\nBEFORE note:', scores['X']['note'])
print('AFTER note: ', scores['Y']['note'])

CHECK                            BEFORE       AFTER
---------------------------------------------------
words (<=120)                       260          60
bullets (==3)                        13           3
jargon hits (==0)                     6           0
jargon words               churn, gmv, gross merchandise, reconciliation, sell-through, sla        none
numbers not in memo (==0)             2           0
which numbers                    10, 50        none
BEFORE: 10, 50
AFTER: none
CRITERION      BEFORE    AFTER   DELTA
--------------------------------------
relevance           3        5      +2  <- module metric
coherence           5        5      +0  <- module metric
accuracy            5        2      -3  <- module metric
length              5        5      +0
format              1        5      +4
audience            3        5      +2
--------------------------------------
TOTAL /30          22       27      +5

BEFORE note: Failed format constraints and included too much

## 7. RSIP - does a second pass keep paying?

**Technique: Recursive Self-Improvement Prompting.** The module defines RSIP as the model evaluating
its own prior response, applying corrections, and generating a new version, recursively. Section 3
was pass one. This is pass two, the identical reflection prompt run against the already-revised
summary.

The interesting result is not "it got better again." It is whether the curve flattens. If pass two
produces no measurable gain, that is worth knowing before anyone builds a loop that burns three API
calls per summary forever.

In [14]:
reflection_2 = ask(
    PROMPT_REFLECT.format(memo=MEMO, draft=summary_after),
    system=REFLECT_SYSTEM,
)
parts_2 = re.split(r'REVISED SUMMARY:\s*', reflection_2, maxsplit=1)
assert len(parts_2) == 2, 'header not found on pass 2'
critique_2, summary_pass2 = parts_2[0].replace('CRITIQUE:', '').strip(), parts_2[1].strip()

print('PASS 2 CRITIQUE')
print('-' * 70)
print(critique_2)
print()
print('PASS 2 SUMMARY')
print('-' * 70)
print(summary_pass2)

# Same mechanical checks, all three versions.
rows3 = [check(summary_before, 'BEFORE'), check(summary_after, 'PASS 1'), check(summary_pass2, 'PASS 2')]
keys = list(rows3[0].keys())[1:]
w = max(len(k) for k in keys) + 2
print(f"{'CHECK':<{w}}{'BEFORE':>12}{'PASS 1':>12}{'PASS 2':>12}")
print('-' * (w + 36))
for k in keys:
    print(f'{k:<{w}}' + ''.join(f'{str(r[k]):>12}' for r in rows3))

print()
scores_2 = grade(summary_after, summary_pass2)
t1, t2 = score_table(scores_2, col_a='PASS 1', col_b='PASS 2')
print()
gain_1 = total_after - total_before
gain_2 = t2 - t1
print(f'graded gain, pass 1: {gain_1:+d} points')
print(f'graded gain, pass 2: {gain_2:+d} points')
print('Recursion is still paying.' if gain_2 >= gain_1 and gain_2 > 0
      else 'Diminishing returns: pass 1 did the work, pass 2 mostly reshuffles wording.')

def block(title, text):
    print('=' * 74)
    print(title)
    print('=' * 74)
    print(text)
    print()

block('BEFORE - loose prompt', summary_before)
block('THE CRITIQUE', critique)
block('AFTER - revised against the five criteria', summary_after)
block('PASS 2 - RSIP, a second reflection on the revision', summary_pass2)

PASS 2 CRITIQUE
----------------------------------------------------------------------
1. PASS
2. FAIL: "231" is in the memo, but the summary implies it is the costliest problem, which is an inference not stated in the memo.
3. FAIL: 68 words. (Requirement: 120 words maximum).
4. FAIL: Did not include the required number behind the "why" in the bullets.
5. PASS

PASS 2 SUMMARY
----------------------------------------------------------------------
* Hire a third photographer at 1st Street during move-in surges to prevent listing delays, as the current staff missed 231 items.
* Require a second measurement entry or template at intake to reduce fit-related returns, which totaled 41 items.
* Automate return checks against payouts to stop late payments, which caused delays in 3 cycles.

The costliest problem is late listings at 231.
CHECK                            BEFORE      PASS 1      PASS 2
---------------------------------------------------------------
words (<=120)                   